In [2]:
import pdb

from torchvision.models.detection.faster_rcnn import TwoMLPHead, FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNHeads, MaskRCNNPredictor
from torchvision.models.detection.anchor_utils import AnchorGenerator

#from torchvision.models.detection.roi_heads import RoIHeads
import sys
sys.path.insert(0, '../')
from utils.roiheads import RoIHeads

from torchvision.models.detection.rpn import RPNHead, RegionProposalNetwork
from torchvision.ops.poolers import MultiScaleRoIAlign
from torchvision.models.detection import backbone_utils
from torchvision.models.detection.generalized_rcnn import GeneralizedRCNN
from torchvision.models.detection.transform import GeneralizedRCNNTransform
import torch
from torchvision.ops import FeaturePyramidNetwork

from models.generalized_rcnn import GeneralizedRCNN
import pytorch_lightning as pl
import lightning as L

import os
import sys
sys.path.append(os.path.join(os.getcwd(), *tuple(['..'])))
import argparse

from typing import Callable, Dict, List, Optional, Set
from collections import OrderedDict
import pdb
import torch
from torch import nn, Tensor
import torch.optim
#import wandb
from model_mrcnn import _default_mrcnn_config, build_default
from features import build_features
#from features import transforms as T
from utils.engine import evaluate
import torchvision
import matplotlib.pyplot as plt
from visualization.explain import ExplainPredictions
import pandas as pd
import plotly.graph_objects as go
import pdb
from sklearn.metrics import precision_recall_curve, auc
import numpy as np
from torchvision import transforms

from collections import OrderedDict
import torch
from torch import nn, Tensor
import warnings
from typing import Tuple, List, Dict, Optional, Union

from lightning.pytorch.loggers import WandbLogger

from torchmetrics.classification import MulticlassConfusionMatrix
import torchvision.ops.boxes as bops
from torchmetrics.classification import Dice
from lightning.pytorch.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import WandbLogger
import wandb


#from lightning.pytorch import loggers as pl_loggers


In [3]:
print("Using torch", torch.__version__)

Using torch 2.0.0


In [4]:
class GeneralizedRCNN(nn.Module):
    """
    Main class for Generalized R-CNN.

    Args:
        backbone (nn.Module):
        rpn (nn.Module):
        roi_heads (nn.Module): takes the features + the proposals from the RPN and computes
            detections / masks from it.
        transform (nn.Module): performs the data transformation from the inputs to feed into
            the model
    """

    def __init__(self, rpn, roi_heads):
        super(LitMaskRCNN, self).__init__()
        self.transform = transform
        self.backbone = backbone
        self.rpn = rpn
        self.roi_heads = roi_heads
        # used only on torchscript mode
        self._has_warned = False

    @torch.jit.unused
    def eager_outputs(self, losses, detections):
        # type: (Dict[str, Tensor], List[Dict[str, Tensor]]) -> Union[Dict[str, Tensor], List[Dict[str, Tensor]]]
        if self.training:
            return losses

        return detections

    def forward(self, images, targets=None):
        # type: (List[Tensor], Optional[List[Dict[str, Tensor]]]) -> Tuple[Dict[str, Tensor], List[Dict[str, Tensor]]]
        """
        Args:
            images (list[Tensor]): images to be processed
            targets (list[Dict[Tensor]]): ground-truth boxes present in the image (optional)

        Returns:
            result (list[BoxList] or dict[Tensor]): the output from the model.
                During training, it returns a dict[Tensor] which contains the losses.
                During testing, it returns list[BoxList] contains additional fields
                like `scores`, `labels` and `mask` (for Mask R-CNN models).

        """
        if self.training and targets is None:
            raise ValueError("In training mode, targets should be passed")
        if self.training:
            assert targets is not None
            for target in targets:
                boxes = target["boxes"]
                if isinstance(boxes, torch.Tensor):
                    if len(boxes.shape) != 2 or boxes.shape[-1] != 4:
                        raise ValueError("Expected target boxes to be a tensor"
                                         "of shape [N, 4], got {:}.".format(
                                             boxes.shape))
                else:
                    raise ValueError("Expected target boxes to be of type "
                                     "Tensor, got {:}.".format(type(boxes)))

        #type hint
        original_image_sizes: List[Tuple[int, int]] = []
        for img in images:
            val = img.shape[-2:]
            assert len(val) == 2
            original_image_sizes.append((val[0], val[1]))

        #TODO Why Another Transform Here?
        images, targets = self.transform(images, targets)
        

        # Check for degenerate boxes
        # TODO: Move this to a function
        if targets is not None:
            for target_idx, target in enumerate(targets):
                boxes = target["boxes"]
                degenerate_boxes = boxes[:, 2:] <= boxes[:, :2]
                if degenerate_boxes.any():
                    print(target_idx)
                    print(target["boxes"])
                    pdb.set_trace()
                    # print the first degenerate box
                    bb_idx = torch.where(degenerate_boxes.any(dim=1))[0][0]
                    degen_bb: List[float] = boxes[bb_idx].tolist()
                    raise ValueError("All bounding boxes should have positive height and width."
                                     " Found invalid box {} for target at index {}."
                                     .format(degen_bb, target_idx))

        # Image is passed through backbone model
        features = self.backbone(images.tensors)
        #self.visualize_feature_maps(images, features, show=False)

        if isinstance(features, torch.Tensor):
            features = OrderedDict([('0', features)])

        # Features - odict_keys(['0', '1', '2', '3', 'pool'])
        # targets - dict_keys(['boxes', 'labels', 'masks', 'image_id', 'area'])
        # images - torch.Size([3, 3, 1024, 1024])
        # proposals - torch.Size([2000, 4])
        proposals, proposal_losses = self.rpn(images, features, targets)
        self.visualize_rpn_proposals(images, proposals, False)
        detections, detector_losses = self.roi_heads(features, proposals, images.image_sizes, targets)
        detections = self.transform.postprocess(detections, images.image_sizes, original_image_sizes)

        if len(detections)!= 0:
            self.visualize_roi_detections(images, detections, 20,False)

        losses = {}
        losses.update(detector_losses)
        losses.update(proposal_losses)

        if torch.jit.is_scripting():
            if not self._has_warned:
                warnings.warn("RCNN always returns a (Losses, Detections) tuple in scripting")
                self._has_warned = True
            return losses, detections
        else:
            return self.eager_outputs(losses, detections)

In [5]:
class _default_mrcnn_config:
    def __init__(
        self,
        num_classes=91,
        backbone_num_features=5,
        backbone_out_channels=256,
    ):

        # self.num_classes = 91 if num_classes is None else num_classes
        # self.backbone_num_features = 5 if backbone_num_features is None else backbone_num_features
        # self.backbone_out_channels = 256 if backbone_out_channels is None else backbone_out_channels


        anchor_config = dict(
            sizes=[2 ** i for i in range(5, 5 + backbone_num_features)],
            # scales=[2 ** i for i in range(-1, 1 + 1)], # based on original paper variant
            scales=[2 ** i for i in range(0, 0 + 1)], # default torch implementation variant
            ratios=[2 ** i for i in range(-1, 1 + 1)],
        )
        rpn_head_config = dict(
            in_channels=backbone_out_channels,
            num_anchors=len(anchor_config['scales']) * len(anchor_config['ratios']), #
            # conv_depth=1, # option unsupported by some versions
        )
        rpn_config = dict(
            fg_iou_thresh=0.7,
            bg_iou_thresh=0.3,
            batch_size_per_image=256,
            positive_fraction=0.5,
            pre_nms_top_n=dict(
                training=2000,
                testing=1000,
            ),
            post_nms_top_n=dict(
                training=2000,
                testing=1000,
            ),
            nms_thresh=0.7,
            score_thresh=0.0,
        )
        box_roi_pool_config = dict(
            featmap_names=[str(i) for i in range(4)],
            output_size=7,
            sampling_ratio=2,
            canonical_scale=224, #
            canonical_level=4, #
        )
        box_head_config = dict(
            in_channels=backbone_out_channels * (box_roi_pool_config['output_size'] ** 2),
            representation_size=1024, #1024
        )
        box_predictor_config = dict(
            in_channels=box_head_config['representation_size'],
            num_classes=num_classes,
        )
        mask_roi_pool_config = dict(
            featmap_names=box_roi_pool_config['featmap_names'],
            output_size=14,
            sampling_ratio=2,
            canonical_scale=box_roi_pool_config['canonical_scale'],
            canonical_level=box_roi_pool_config['canonical_level'],
        )
        mask_head_config = dict(
            in_channels=backbone_out_channels,
            layers=tuple([256 for _ in range(4)]),
            dilation=1,
        )
        mask_predictor_config = dict(
            in_channels=mask_head_config['layers'][-1],
            dim_reduced=256,
            num_classes=num_classes,
        )
        roi_heads_config = dict(
            fg_iou_thresh=0.5,
            bg_iou_thresh=0.5,
            batch_size_per_image=512,
            positive_fraction=0.25,
            bbox_reg_weights=None, # appears to use (10., 10., 5., 5.,) by default?
            score_thresh=0.05,
            nms_thresh=0.5,
            detections_per_img=100,
        )

        self.config = dict(
            rpn_config=dict(
                anchor_config=anchor_config,
                rpn_head_config=rpn_head_config,
                rpn_config=rpn_config,
            ),
            roi_heads_config=dict(
                box_config=dict(
                    box_roi_pool_config=box_roi_pool_config,
                    box_head_config=box_head_config,
                    box_predictor_config=box_predictor_config,
                ),
                mask_config=dict(
                    mask_roi_pool_config=mask_roi_pool_config,
                    mask_head_config=mask_head_config,
                    mask_predictor_config=mask_predictor_config,
                ),
                roi_heads_config=roi_heads_config,
            ),
        )

def defaults(d, keys, prefix=None, suffix=None, list=False):
    if prefix is not None:
        keys = [f'{prefix}{k}' for k in keys]
    if suffix is not None:
        keys = [f'{k}{suffix}' for k in keys]

    assert set(d.keys()).issuperset(set(keys))
    values = [(k, d[k]) for k in keys]
    if list:
        return [k for _, k in values]
    return dict(values)


def build_anchor_generator(config):
    keywords = 'sizes scales ratios'
    sizes, scales, ratios = defaults(config, keywords.split(), list=True)

    sizes_t = tuple([tuple([size * scale for scale in scales]) for size in sizes])
    ratios_t = tuple([tuple(ratios) for _ in sizes])

    return AnchorGenerator(
        sizes=sizes_t,
        aspect_ratios=ratios_t,
    )

def build_rpn_head(config):
    keywords = 'in_channels num_anchors' # 'in_channels num_anchors conv_depth'
    config = defaults(config, keywords.split())

    return RPNHead(**config)

def build_rpn(anchor_config, rpn_head_config, rpn_config):
    keywords = 'fg_iou_thresh bg_iou_thresh batch_size_per_image positive_fraction pre_nms_top_n post_nms_top_n nms_thresh score_thresh'
    rpn_config = defaults(rpn_config, keywords.split())
    for k in 'pre post'.split(): # sub-config check to ensure pre, post nms configs properly structured, as expected by RegionProposalNetwork()
        k = f'{k}_nms_top_n'
        rpn_config[k] = defaults(rpn_config[k], 'training testing'.split())
        # rpn_config[k] = defaults(rpn_config[k], 'train test'.split())
    return RegionProposalNetwork(
        anchor_generator=build_anchor_generator(anchor_config),
        head=build_rpn_head(rpn_head_config),
        **rpn_config,
    )

def build_roi_pool(roi_pool_config):
    keywords = 'featmap_names output_size sampling_ratio canonical_scale canonical_level'
    roi_pool_config = defaults(roi_pool_config, keywords.split())

    return MultiScaleRoIAlign(**roi_pool_config)

def build_box_head(box_head_config):
    keywords = 'in_channels representation_size'
    box_head_config = defaults(box_head_config, keywords.split())

    return TwoMLPHead(**box_head_config)

def build_box_predictor(box_predictor_config):
    keywords = 'in_channels num_classes'
    box_predictor_config = defaults(box_predictor_config, keywords.split())

    return FastRCNNPredictor(**box_predictor_config)

def build_mask_head(mask_head_config):
    keywords = 'in_channels layers dilation'
    mask_head_config = defaults(mask_head_config, keywords.split())

    return MaskRCNNHeads(**mask_head_config)

def build_mask_predictor(mask_predictor_config):
    keywords = 'in_channels dim_reduced num_classes'
    mask_predictor_config = defaults(mask_predictor_config, keywords.split())

    return MaskRCNNPredictor(**mask_predictor_config)

def build_box_heads(box_roi_pool_config, box_head_config, box_predictor_config):
    return dict(
        box_roi_pool=build_roi_pool(box_roi_pool_config),
        box_head=build_box_head(box_head_config),
        box_predictor=build_box_predictor(box_predictor_config),
    )

def build_mask_heads(mask_roi_pool_config, mask_head_config, mask_predictor_config):
    return dict(
        mask_roi_pool=build_roi_pool(mask_roi_pool_config),
        mask_head=build_mask_head(mask_head_config),
        mask_predictor=build_mask_predictor(mask_predictor_config),
    )

def build_roi_heads(box_config, mask_config, roi_heads_config):
    keywords = 'fg_iou_thresh bg_iou_thresh batch_size_per_image positive_fraction bbox_reg_weights score_thresh nms_thresh detections_per_img'
    head_keywords = 'roi_pool head predictor'.split()

    roi_heads_config = defaults(roi_heads_config, keywords.split())

    box_config = defaults(box_config, head_keywords, prefix='box_', suffix='_config')
    box_heads = build_box_heads(**box_config)

    mask_heads = dict()
    if mask_config is not None:
        mask_config = defaults(mask_config, head_keywords, prefix='mask_', suffix='_config')
        mask_heads = build_mask_heads(**mask_config)

    return RoIHeads(
        **box_heads,
        **roi_heads_config,
        **mask_heads,
    )


def build_default(config, im_size=1024, backbone=None, transform=None):
    # TODO:
    #   import custom backbone
    #   import custom GeneralizedRCNNTransform
    #   consolidate backbone, transform configs with default configs


    rpn_config, roi_heads_config = defaults(config, 'rpn roi_heads'.split(), suffix='_config', list=True) # list=True flag unpacks the keys rather than returning a dict
    roi_heads_config = defaults(roi_heads_config, 'box mask roi_heads'.split(), suffix='_config')

    rpn = build_rpn(**rpn_config)
    roi_heads = build_roi_heads(**roi_heads_config)
    if backbone is None:
        backbone = backbone_utils.resnet_fpn_backbone(
            'resnet152',
            pretrained=True,
            trainable_layers=3
            )
    else:
        backbone 
    """ 
    # load dino model
    dinov2_vits14 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
    #print(backbone)
    #print(dinov2_vits14)
    backbone = dinov2_vits14
    """
    #fpn_dinov2 = FeaturePyramidNetwork(dinov2_vits14, 100)
    if transform is None:
        transform = GeneralizedRCNNTransform(
            min_size=im_size,
            max_size=im_size,            #size_divisible=14,
            #image_mean=[0.485, 0.456, 0.406],
            #image_std=[0.229, 0.224, 0.225],
            #image_mean=[0.8753, 0.8724, 0.8949], ## for original image/labels
            #image_std=[0.0439, 0.0443, 0.0403],## for original image/labels
            #image_mean=[0.8655, 0.8491, 0.8572], # for balanced data
            #image_std=[0.0535, 0.0616, 0.0664], # for balanced data
            #image_mean = [0.8198, 0.7572, 0.7618],# for negclassadded  data
            #image_std =  [0.0848, 0.1066, 0.1211] # for negclassadded  data
            image_mean= [0.8224, 0.7101, 0.7279], #for train_br
            image_std = [0.0940, 0.2354, 0.2226]
            #image_mean=[0.8318, 0.7590, 0.7785], not sure which is this?
            #image_std=[0.0898, 0.1413, 0.1368]
            #image_mean= [0.8086, 0.7433, 0.7586], # for br_all
            #image_std=[0.0949, 0.1241, 0.1288]
        )
    else:
        transform
    return backbone, rpn, roi_heads, transform
    #return GeneralizedRCNN(backbone, rpn, roi_heads, transform)


In [6]:
class_names = ["True","Pre","False"]
def get_outputs(outputs, threshold):
    mask_list = []
    label_list = []
    for j in range(len(outputs)):
        scores = list(outputs[j]['scores'].detach().cpu().numpy())
        # print("\n scores", max(scores))
        # index of those scores which are above a certain threshold
        thresholded_preds_inidices = [scores.index(i) for i in scores if i > threshold]
        #print(thresholded_preds_inidices)
        scores = [scores[x] for x in thresholded_preds_inidices]
        # get the masks
        masks = (outputs[j]['masks']>0.5).squeeze().detach().cpu().numpy()
        # print("masks", masks)
        # discard masks for objects which are below threshold
        masks = [masks[x] for x in thresholded_preds_inidices]
        # get the bounding boxes, in (x1, y1), (x2, y2) format
        boxes = [[(int(i[0]), int(i[1])), (int(i[2]), int(i[3]))]  for i in outputs[j]['boxes'].detach().cpu()]
        # discard bounding boxes below threshold value
        boxes = [boxes[x] for x in thresholded_preds_inidices]
        # get the classes labels
        # print('labels', outputs[0]['labels'])
        #print(outputs[0]['labels'])
        #print(thresholded_preds_count)
        #print(outputs[0]['labels'])
        labels = [class_names[i-1] for i in outputs[j]['labels']]
        #labels = [i for i in outputs[0]['labels']]
        #print(labels)
        labels = [labels[x] for x in thresholded_preds_inidices]
        mask_list.append(masks)
        label_list.append(labels)
    return mask_list, label_list

In [11]:
def match_mask(masked_image,binary_array):
    #num_classes = len(set(list(np.unique(masked_image)) +  list(np.unique(binary_array))))
    num_classes = 2
    metric = MulticlassConfusionMatrix(num_classes=num_classes).to(device)
    conf_final=torch.tensor(np.zeros((num_classes,num_classes))).to(device).to(torch.int64)
    for i in range(len(masked_image)):
        preds = torch.tensor(masked_image[i]).to(device).to(torch.int64)
        target = torch.tensor(binary_array[i]).to(device).to(torch.int64)
        a1 = metric(preds,target)
        conf_final = conf_final + a1     
    conf_final_np = conf_final.cpu().numpy()
    #print(conf_final_np)
    total_predicted = np.sum(conf_final_np, axis=0) 
    diag_elements = np.diag(conf_final_np)
    precision = diag_elements/total_predicted
    total_actual = np.sum(conf_final_np, axis=1) 
    recall = diag_elements/total_actual
    f1_score = (2*precision*recall)/(recall+precision)
    iou_coeff = (diag_elements)/(total_predicted+total_actual-diag_elements)
    #csv_filename_tosave = "Eval_Metric_"+ geofile.split(".")[0] + ".csv"
    eval_metrics = pd.DataFrame({"Class":["True/Pre","False"],"Precision":precision, "recall":recall,"f1_score":f1_score,"iou_coeff":iou_coeff})
    #eval_metrics.to_csv(os.path.join(eval_dir,csv_filename_tosave))
    return eval_metrics[eval_metrics["Class"]=="False"]["f1_score"].values[0]


def match_label(pred_label, gt_label):
    if pred_label==gt_label:
        return 1
    else:
        return 0

def actual_label_target(gt_label):
    if (gt_label.cpu().numpy()==1):
        return "True"
    if (gt_label.cpu().numpy()==2):
        return "Pre"
    return None


def evaluate_metrics(target,masks, labels):
    f1_score_list=[]
    matched_label_list=[]
    mean_f1_score = -1
    mean_matched_label=-1
    for i in range(len(target)):
        target_label = actual_label_target(target[i]['labels'])
        #print(target[i]['masks'][0].shape, masks[0].shape)
        for j in range(len(masks)):
            for k in range(len(masks[j])):
                target_mask = target[i]['masks'][0].cpu().numpy()
                target_mask= np.where(target_mask > 0, 1, 0)
                if target_mask.shape==masks[j][k].shape:
                    f1_score = match_mask(masks[j][k],target_mask)
                    f1_score_list.append(f1_score)
                    if f1_score>0:
                        matched_label = match_label(labels[j][k],target_label)
                        matched_label_list.append(matched_label)
                    else:
                        matched_label_list.append(0)
    if len(f1_score_list)>0:
        mean_f1_score=np.nansum(f1_score_list)/len(f1_score_list)
    if len(matched_label_list)>0:
        mean_matched_label = sum(matched_label_list)/len(matched_label_list)
    #print(f1_score_list, matched_label_list)
    return mean_f1_score, mean_matched_label

In [12]:
class LitMaskRCNN(L.LightningModule):
    def __init__(self, optim_config,backbone,rpn,roi_heads,transform):
        super().__init__()
        #self.model_config = _default_mrcnn_config(num_classes=1 + train_config['num_classes']).config
        #self.model = model
        self.optim_config = optim_config
        #self.model = build_default(model_config, im_size=1024)
        self.loss_names = 'objectness rpn_box_reg classifier box_reg mask'.split()
        self.loss_weights = [1., 4., 1., 4., 1.,]
        self.loss_weights = OrderedDict([(f'loss_{name}', weight) for name, weight in zip(self.loss_names, self.loss_weights)])
        self.backbone=backbone
        self.rpn = rpn
        self.roi_heads = roi_heads
        self.transform = transform
        # used only on torchscript mode
        self._has_warned = False
        self.automatic_optimization = False
        self.save_hyperparameters()
        self.avg_segmentation_overlap = 0.0
        self.val_acc = 0.0

    @torch.jit.unused
    def eager_outputs(self, losses, detections):
        # type: (Dict[str, Tensor], List[Dict[str, Tensor]]) -> Union[Dict[str, Tensor], List[Dict[str, Tensor]]]
        if self.training:
            return losses

        return detections

    def forward(self, images, targets=None):
        # type: (List[Tensor], Optional[List[Dict[str, Tensor]]]) -> Tuple[Dict[str, Tensor], List[Dict[str, Tensor]]]
        """
        Args:
            images (list[Tensor]): images to be processed
            targets (list[Dict[Tensor]]): ground-truth boxes present in the image (optional)

        Returns:
            result (list[BoxList] or dict[Tensor]): the output from the model.
                During training, it returns a dict[Tensor] which contains the losses.
                During testing, it returns list[BoxList] contains additional fields
                like `scores`, `labels` and `mask` (for Mask R-CNN models).

        """
        if self.training and targets is None:
            raise ValueError("In training mode, targets should be passed")
        if self.training:
            assert targets is not None
            for target in targets:
                boxes = target["boxes"]
                if isinstance(boxes, torch.Tensor):
                    if len(boxes.shape) != 2 or boxes.shape[-1] != 4:
                        raise ValueError("Expected target boxes to be a tensor"
                                         "of shape [N, 4], got {:}.".format(
                                             boxes.shape))
                else:
                    raise ValueError("Expected target boxes to be of type "
                                     "Tensor, got {:}.".format(type(boxes)))

        #type hint
        original_image_sizes: List[Tuple[int, int]] = []
        for img in images:
            val = img.shape[-2:]
            assert len(val) == 2
            original_image_sizes.append((val[0], val[1]))

        #TODO Why Another Transform Here?
        images, targets = self.transform(images, targets)
        

        # Check for degenerate boxes
        # TODO: Move this to a function
        if targets is not None:
            for target_idx, target in enumerate(targets):
                boxes = target["boxes"]
                degenerate_boxes = boxes[:, 2:] <= boxes[:, :2]
                if degenerate_boxes.any():
                    print(target_idx)
                    print(target["boxes"])
                    pdb.set_trace()
                    # print the first degenerate box
                    bb_idx = torch.where(degenerate_boxes.any(dim=1))[0][0]
                    degen_bb: List[float] = boxes[bb_idx].tolist()
                    raise ValueError("All bounding boxes should have positive height and width."
                                     " Found invalid box {} for target at index {}."
                                     .format(degen_bb, target_idx))

        # Image is passed through backbone model
        features = self.backbone(images.tensors)
        #self.visualize_feature_maps(images, features, show=False)

        if isinstance(features, torch.Tensor):
            features = OrderedDict([('0', features)])

        # Features - odict_keys(['0', '1', '2', '3', 'pool'])
        # targets - dict_keys(['boxes', 'labels', 'masks', 'image_id', 'area'])
        # images - torch.Size([3, 3, 1024, 1024])
        # proposals - torch.Size([2000, 4])
        proposals, proposal_losses = self.rpn(images, features, targets)
        #self.visualize_rpn_proposals(images, proposals, False)
        detections, detector_losses = self.roi_heads(features, proposals, images.image_sizes, targets)
        detections = self.transform.postprocess(detections, images.image_sizes, original_image_sizes)

        #if len(detections)!= 0:
            #self.visualize_roi_detections(images, detections, 20,False)

        losses = {}
        losses.update(detector_losses)
        losses.update(proposal_losses)

        if torch.jit.is_scripting():
            if not self._has_warned:
                warnings.warn("RCNN always returns a (Losses, Detections) tuple in scripting")
                self._has_warned = True
            return losses, detections
        else:
            return self.eager_outputs(losses, detections)
        
        
        
    def get_loss_fn(self, weights, default=0.):
        def compute_loss_fn(losses):
            item = lambda k: (k, losses[k].item())
            metrics = OrderedDict(list(map(item, [k for k in weights.keys() if k in losses.keys()] + [k for k in losses.keys() if k not in weights.keys()])))
            loss = sum(map(lambda k: losses[k] * (weights[k] if weights is not None and k in weights.keys() else default), losses.keys()))
            return loss, metrics
        return compute_loss_fn
    
    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        #print(batch)
        opt = self.optimizers()
        images, targets = batch 
        #images = [image for image in images]
        #targets = [dict([(k, v) for k, v in target.items()]) for target in targets]
        opt.zero_grad()
        loss_fn = self.get_loss_fn(self.loss_weights)
        loss, metrics = loss_fn(self.forward(images, targets))
        
        #loss.backward()
        self.manual_backward(loss)
        opt.step()
        #log_metrics.append(dict(epoch=epoch, loss=loss.item(), metrics=metrics))
        print_logs = "batch no : {batch_no}, total loss : {loss},  classifier :{classifier}, mask: {mask} ==================="
        print(print_logs.format( batch_no=batch_idx, loss=loss.item(),  classifier=metrics['loss_classifier'], mask=metrics['loss_mask']))
        #yield log_metrics
    
    #def backward(self, loss):
    #    loss.backward()
    
    def configure_optimizers(self):
        optimizer = self.optim_config['cls']([dict(params=list(self.parameters()))], **self.optim_config['defaults'])
        return optimizer
    
    def validation_step(self, batch, batch_idx):
        # this is the validation loop
        images, targets = batch 
        #images = [image for image in batch[0]]
        #targets = [dict([(k, v) for k, v in target.items()]) for target in batch[1]]
        #loss_fn = self.get_loss_fn(self.loss_weights)
        outputs = self.forward(images, targets)
        masks, labels = get_outputs(outputs, 0.10)
        f1_mean, labels_matched =  evaluate_metrics(targets, masks, labels)
        self.avg_segmentation_overlap = f1_mean
        self.val_acc = labels_matched
        if (f1_mean>=0) or (labels_matched>=0):
            print(f1_mean, labels_matched)
        return f1_mean, labels_matched
        #outputs1 = [x for x in outputs if len(x["labels"])!=0]
        #if len(outputs1)>0:
        #    print(outputs1[0].keys())
        #loss, metrics = loss_fn(outputs1[0])
        #loss, metrics = loss_fn(self.forward(images, targets))
        #print_logs = "batch no : {batch_no}, total loss : {loss},  classifier :{classifier}, mask: {mask} ==================="
        #print(print_logs.format( batch_no=batch_idx, loss=loss.item(),  classifier=metrics['loss_classifier'], mask=metrics['loss_mask']))
        
    
    def test_step(self, batch, batch_idx):
        # this is the validation loop
        images, targets = batch 
        #images = [image for image in batch[0]]
        #targets = [dict([(k, v) for k, v in target.items()]) for target in batch[1]]
        #loss_fn = self.get_loss_fn(self.loss_weights)
        #loss, metrics = loss_fn(self.forward(images, targets))
        outputs = self.forward(images)
        masks, labels = get_outputs(outputs, 0.10)
        f1_mean, labels_matched =  evaluate_metrics(targets, masks, labels)
        return f1_mean, labels_matched
    

In [13]:
def train_model(wandb_config,train_config,model_config,optim_config, dataset_base_dir, dataset_train_location, dataset_test_location  ):
    run = wandb.init(**wandb_config)
    run_id, run_dir = run.id, run.dir
    wandb_logger = WandbLogger()
    collate_fn=lambda x: tuple(zip(*x))

    data_transforms = transforms.Compose([
            transforms.ToTensor(),
        ])
    train_dataset = build_features.LBD_Dataset(dataset_train_location, data_transforms,["images","labels"])

    train_data_loader = torch.utils.data.DataLoader(
                train_dataset, batch_size=train_config['batch_size'], shuffle=True, num_workers=4,
                collate_fn=collate_fn)

    val_data_loader = torch.utils.data.DataLoader(
                train_dataset, batch_size=train_config['batch_size'], shuffle=False, num_workers=4,
                collate_fn=collate_fn)

    

    
    backbone, rpn, roi_heads, transform1 = build_default(model_config, im_size=1024)

    model  = LitMaskRCNN(optim_config,backbone,rpn,roi_heads,transform1)

    #ckpt_path="/gladstone/finkbeiner/steve/work/data/npsad_data/monika/models/checkpoints/my_checkpoint.ckpt"
    #model_ckpt = 
    chkpt = ModelCheckpoint(monitor="val_acc", mode="max")

    #tensorboard = pl_loggers.TensorBoardLogger(save_dir="/gladstone/finkbeiner/steve/work/data/npsad_data/monika/models/")

    trainer = L.Trainer(limit_train_batches=6, max_epochs=train_config['epochs'],devices=1, accelerator="gpu",default_root_dir = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/pytorch_lightning_model_output-{run_id}",num_sanity_val_steps=0,
                        check_val_every_n_epoch=5,callbacks=[chkpt])
    train_loader = train_data_loader
    valid_loader = val_data_loader
    trainer.fit(model, train_loader, valid_loader) 
    return Model.load_from_checkpoint(chkpt.best_model_path)


In [14]:
train_config = dict(
    epochs = 10 ,
    batch_size = 3,
    num_classes = 3,
    device_id = 0,
    ckpt_freq =500,
    eval_freq = 25,
)
model_config = _default_mrcnn_config(num_classes=1 + train_config['num_classes']).config
optim_config = dict(
            cls=torch.optim.Adam,
            defaults=dict(lr=0.0001,weight_decay=1e-5) 
        )
wandb_config = dict(
        project='LBD',
        entity='monika-ahirwar',
        config=dict(
            train_config=train_config,
            model_config=model_config,
            optim_config=optim_config,
        ),
        save_code=False,
        group='runs',
        job_type='train',
    )
#run = wandb.init(**wandb_config)
#run_id, run_dir = run.id, run.dir

dataset_base_dir = '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/'
dataset_train_location = '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/train_negclass_allBR'
dataset_test_location = '/home/mahirwar/Desktop/Monika/npsad_data/monika/LBD/Train_val_LB/val_negclass_allBR'

device = torch.device('cpu')
if torch.cuda.is_available():
    assert train_config['device_id'] >= 0 and train_config['device_id'] < torch.cuda.device_count()
    device = torch.device('cuda', train_config['device_id'])

best_mode = train_model(wandb_config,train_config,model_config,optim_config, dataset_base_dir, dataset_train_location, dataset_test_location  )

/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:135: UserWarning: Using 'backbone_name' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet152_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet152_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/home/mahir

Epoch 4: 100%|██████████| 6/6 [00:04<00:00,  1.31it/s, v_num=2]

/tmp/ipykernel_2555684/4229084679.py:18: RuntimeWarning: invalid value encountered in true_divide
  f1_score = (2*precision*recall)/(recall+precision)


0.0 0.0
0.24301675977653628 0.3333333333333333
0.0 0.0
0.21725333840823524 0.0
0.0 0.0
0.20928865373309816 0.3333333333333333
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.0 0.0
0.083746515426521 0.16666666666666666
0.09946622829741529 0.14285714285714285
0.0493855050052033 0.07777777777777778
0.04321705389832072 0.0875
0.057309805846017636 0.10476190476190476
0.16210424081918434 0.3148148148148148
0.004894653150467104 0.0
0.005722525190588734 0.0
0.14317549647798938 0.2222222222222222
0.0642823032575619 0.13450292397660818
0.09057827511484305 0.10144927536231885
0.027368073179436788 0.05291005291005291
0.11448157037408102 0.19230769230769232
0.046492001845734106 0.09722222222222222
0.07298631869222365 0.09259259259259259
0.09275544087642215 0.1111111111111111
0.04900236245882883 0.038461538461538464
0.051062805173018554 0.08333333333333333
0.06671223399748308 0.0
0.17385277891563264 0.009009009009009009
0.2124414181782527 0.08333333333333333
0.24048593068822288 0.3333333333333333
0.24659406556940

KeyboardInterrupt: 

: 

In [15]:
## CONFIGS ##
#collate_fn = lambda _: tuple(zip(*_)) # one-liner, no need to import
#run = wandb.init(**wandb_config)
#assert run is wandb.run # run was successfully initialized, is not None
#run_id, run_dir = run.id, run.dir
#exp_name = run.name

#artifact_name = f'{run_id}-logs'


optim_config = dict(
        cls=torch.optim.Adam,
        defaults=dict(lr=0.0001,weight_decay=1e-5) 
    )

model_config = _default_mrcnn_config(num_classes=1 + train_config['num_classes']).config
backbone, rpn, roi_heads, transform1 = build_default(model_config, im_size=1024)

model  = LitMaskRCNN(optim_config,backbone,rpn,roi_heads,transform1)

ckpt_path="/gladstone/finkbeiner/steve/work/data/npsad_data/monika/models/checkpoints/my_checkpoint.ckpt"
#model_ckpt = 
checkpoint_callback = ModelCheckpoint(monitor="val_acc", mode="max")

#tensorboard = pl_loggers.TensorBoardLogger(save_dir="/gladstone/finkbeiner/steve/work/data/npsad_data/monika/models/")

trainer = L.Trainer(limit_train_batches=6, max_epochs=train_config['epochs'],devices=1, accelerator="gpu",default_root_dir = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/pytorch_lightning_model_output",num_sanity_val_steps=0,
                    check_val_every_n_epoch=2)
train_loader = train_data_loader
valid_loader = val_data_loader
trainer.fit(model, train_loader, valid_loader) 


/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:135: UserWarning: Using 'backbone_name' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet152_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet152_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/home/mahir

Epoch 5: 100%|██████████| 6/6 [00:08<00:00,  0.74it/s, v_num=49]

/tmp/ipykernel_2553059/712023704.py:35: RuntimeWarning: invalid value encountered in true_divide
  f1_score = (2*precision*recall)/(recall+precision)


0.13658322353974528 0.0
0.14315245478036176 0.16666666666666666
0.14682539682539683 0.16666666666666666
0.0 0.0
0.09599506045774692 0.1111111111111111
0.042894056847545214 0.05555555555555555
0.0 0.0
0.11736474694589878 0.16666666666666666
0.11325219743069642 0.16666666666666666
0.0 0.0
0.1377005347593583 0.16666666666666666
0.148365056124939 0.16666666666666666
0.0 0.0
0.0 0.0
0.07492877492877494 0.08333333333333333
0.09197538812576757 0.1111111111111111
0.13317665491578537 0.16666666666666666
0.0 0.0
0.06056124417686487 0.08333333333333333
0.0 0.0
0.13000373570306845 0.16666666666666666
0.1492216854535695 0.16666666666666666
0.12905231959410243 0.16666666666666666
0.15126958400864396 0.16666666666666666
Epoch 7: 100%|██████████| 6/6 [00:07<00:00,  0.75it/s, v_num=49]

/tmp/ipykernel_2553059/712023704.py:32: RuntimeWarning: invalid value encountered in true_divide
  precision = diag_elements/total_predicted


0.014203533269175903 0.0196078431372549
0.01075701506152108 0.013333333333333334
0.032111768799240485 0.0
0.027790712307393047 0.04954954954954955
0.0 0.0
0.0 0.0
0.044083072166492154 0.0
0.013419626655893888 0.0
0.021516031987139934 0.04093567251461988
0.024586508360654798 0.03571428571428571
0.021978560703602556 0.03597122302158273
0.025132162026888772 0.037953795379537955
0.017505152483074334 0.01839080459770115
0.025848835944056302 0.03305785123966942
0.02163095942122656 0.03333333333333333
0.017808432765263584 0.016304347826086956
0.02270943933024709 0.02412280701754386
0.02623812360764614 0.024054982817869417
0.07650031245487555 0.033854166666666664
0.0058873720136518775 0.008333333333333333
0.04307775686160581 0.057971014492753624
0.0634067490383504 0.08888888888888889
0.01750353504298985 0.015151515151515152
0.05535152051374289 0.0763888888888889
0.006017906942609717 0.009259259259259259
0.018857689346126062 0.02857142857142857
0.04807217677348277 0.015151515151515152
0.0043767

/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/lightning/pytorch/trainer/call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...


In [ ]:
model = LitMaskRCNN.load_from_checkpoint("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/pytorch_lightning_model_output/lightning_logs/version_33/checkpoints/epoch=9-step=60.ckpt")
#print(model.learning_rate)

In [ ]:
model.eval()

In [ ]:
device = torch.device('cpu')
model=model.to(device)

In [ ]:
j=3
scores = list(outputs[3]['scores'].detach().cpu().numpy())
# print("\n scores", max(scores))
# index of those scores which are above a certain threshold
thresholded_preds_inidices = [scores.index(i) for i in scores if i > 0.0525]
#print(thresholded_preds_inidices)
thresholded_preds_count = len(thresholded_preds_inidices)
#print(thresholded_preds_count)
scores = [scores[x] for x in thresholded_preds_inidices]
# get the masks
masks = (outputs[j]['masks']>0.5).squeeze().detach().cpu().numpy()
# print("masks", masks)
# discard masks for objects which are below threshold
masks = [masks[x] for x in thresholded_preds_inidices]
# get the bounding boxes, in (x1, y1), (x2, y2) format
boxes = [[(int(i[0]), int(i[1])), (int(i[2]), int(i[3]))]  for i in outputs[j]['boxes'].detach().cpu()]
# discard bounding boxes below threshold value
boxes = [boxes[x] for x in thresholded_preds_inidices]
# get the classes labels
# print('labels', outputs[0]['labels'])
#print(outputs[0]['labels'])
#print(thresholded_preds_count)
#print(outputs[0]['labels'])
labels = [class_names[i-1] for i in outputs[j]['labels']]
#labels = [i for i in outputs[0]['labels']]
#print(labels)
labels = [labels[x] for x in thresholded_preds_inidices]

In [ ]:
thresholded_preds_inidices

In [ ]:
print(thresholded_preds_count)

In [ ]:

test_config = dict(
        batch_size = 1
    )
#run = ""
test_patient_ids = os.listdir(dataset_test_location)
isKfold_eval = True
device = torch.device('cpu')
if torch.cuda.is_available():
    assert train_config['device_id'] >= 0 and train_config['device_id'] < torch.cuda.device_count()
    device = torch.device('cuda', train_config['device_id'])

if '.DS_Store' in test_patient_ids:
    test_patient_ids.remove('.DS_Store')
    
for t in range(len(test_patient_ids)):
    if len(os.listdir(os.path.join(dataset_test_location,test_patient_ids[t],"images")))==0:
        continue
    test_ds = build_features.LBD_Dataset(os.path.join(dataset_test_location,test_patient_ids[t]), data_transforms,["images","labels"])
    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=test_config['batch_size'], shuffle=False, num_workers=4, collate_fn=collate_fn)
    test_eval_res, test_eval_res_df, full_table = evaluate(run, model, test_loader, device, isKfold_eval)
    #predictions = trainer.test(dataloaders=test_loader)
    #print(predictions)
    break

In [ ]:
import pytorch_lightning as pl
from torch.utils.data import random_split, DataLoader

from torchvision import transforms


class LBDDataModule(pl.LightningDataModule):

    def __init__(self, data_dir: str = './'):
        super().__init__()
        self.data_dir = data_dir
        self.data_transforms = transforms.Compose([
        #transforms.Resize((1022,1022)),
        transforms.ToTensor(),
        
        #transforms.Normalize([0.8753, 0.8724, 0.8949],[0.0439, 0.0443, 0.0403])
    ])

        # self.dims is returned when you call dm.size()
        # Setting default dims here because we know them.
        # Could optionally be assigned dynamically in dm.setup()
        self.dims = (1, 28, 28)

    def prepare_data(self):
        # download
        pass
        

    def setup(self, stage=None):

        # Assign train/val datasets for use in dataloaders
        if stage == 'fit' or stage is None:
            self.train_dataset = build_features.LBD_Dataset(self.data_dir, self.data_transforms,["images","labels"])
            self.val_dataset = build_features.LBD_Dataset(self.data_dir, self.data_transforms,["images","labels"])

            # Optionally...
            # self.dims = tuple(self.mnist_train[0][0].shape)

        # Assign test dataset for use in dataloader(s)
        if stage == 'test' or stage is None:
            self.test_dataset = build_features.LBD_Dataset(self.data_dir, self.data_transforms,["images","labels"])

            # Optionally...
            # self.dims = tuple(self.mnist_test[0][0].shape)

    def train_dataloader(self):
        return torch.utils.data.DataLoader(self.train_dataset, batch_size=train_config['batch_size'], shuffle=True, num_workers=4, collate_fn=collate_fn)

    def val_dataloader(self):
        return torch.utils.data.DataLoader(self.val_dataset, batch_size=train_config['batch_size'], shuffle=False, num_workers=4, collate_fn=collate_fn)

    def test_dataloader(self):
        return torch.utils.data.DataLoader(self.test_dataset, batch_size=test_config['batch_size'], shuffle=False, num_workers=4, collate_fn=collate_fn)

dataset_train_location = '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/train_negclass_allBR'
dm = LBDDataModule(data_dir=dataset_train_location)
dm.setup('fit')
trainer.fit(model, dm)